# Malatya Fault M7.2 Scenario

## 1. Setup

In [ ]:
import os, json, shutil
from pathlib import Path
import xml.etree.ElementTree as ET
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import base64
from IPython.display import HTML

EVENT = 'malatya_fault_se'
os.environ['EVENT'] = EVENT

EVENT_DIR = Path.home() / 'shakemap_profiles/default/data' / EVENT / 'current'
PRODUCTS  = EVENT_DIR / 'products'
GF_OUT    = Path.home() / 'gf_output' / EVENT
TR_DATA   = Path.home() / 'turkey_inputs'

## 2. Run ShakeMap

This scenario uses the **Malatya Fault (Kemaliye Segment)** from the AFAD 2026
Active Fault Database — a 65 km left-lateral fault running N15°E, located
approximately 28 km west of Malatya city.

**Magnitude M7.2** is estimated from the 65 km fault length using the
Wells & Coppersmith (1994) scaling relationship for strike-slip faults.

The same ShakeMap pipeline as the morning demo:
- **`assemble`** — collect inputs (event origin + fault geometry + site conditions)
- **`model`** — compute ground motion using GMMs + Vs30 (~2 min for this extent)
- **`contour`**, **`mapping`**, **`gridxml`** — outputs for visualization and ground failure

In [ ]:
%%bash
shake $EVENT assemble -c 'demo' model contour mapping gridxml

## 3. ShakeMap maps

Because we provided a finite fault geometry, shaking is elongated along the
N15°E fault strike rather than the circular pattern of a point source.
Compare the shaking footprint to the M7.6 San Andreas map from this morning —
note how local geology (Vs30) modifies the pattern.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, name, title in [
    (axes[0], 'intensity.jpg', 'MMI Intensity'),
    (axes[1], 'pga.jpg',       'PGA (%g)'),
]:
    img_path = PRODUCTS / name
    if img_path.exists():
        ax.imshow(mpimg.imread(img_path))
    ax.set_title(title, fontsize=13); ax.axis('off')
plt.suptitle('Malatya Fault M7.2', fontsize=14)
plt.tight_layout(); plt.show()

# save for comparison with Vs30 exercise
shutil.copy(PRODUCTS / 'intensity.jpg',
            Path.home() / 'malatya_global_vs30_intensity.jpg')

**Discussion:**
- Which cities appear to experience the highest shaking?
- How does the shaking pattern compare to the San Andreas scenario from this morning?
- The Malatya Fault is 28 km from Malatya city — how does proximity to the fault
  affect the intensity at the city vs. more distant cities?

## 4. Ground failure

The Malatya region has both steep mountain terrain (landslide prone) and
alluvial river valley soils (liquefaction prone). Run both models:

In [ ]:
%%bash
/opt/conda/envs/gf/bin/gfailbin \
  ~/groundfailure/defaultconfigfiles/models/jessee_2018_slim.ini \
  ~/shakemap_profiles/default/data/$EVENT/current/products/grid.xml \
  --gis -d ~/turkey_inputs

In [ ]:
%%bash
/opt/conda/envs/gf/bin/gfailbin \
  ~/groundfailure/defaultconfigfiles/models/zhu_2017_general_slim.ini \
  ~/shakemap_profiles/default/data/$EVENT/current/products/grid.xml \
  --gis -d ~/turkey_inputs

## 5. Interactive ground failure map

Landslide (left) and liquefaction (right) with shaking contours overlaid.
Download `malatya_gf.html` (Explorer → right-click → Download) to open in a browser.

In [ ]:
%%bash
/opt/conda/envs/gf/bin/python ~/plot_gf_interactive.py \
  --ls-model "Jessee 2018:$HOME/gf_output/$EVENT/${EVENT}_jessee_2018_slim_model.tif:$HOME/groundfailure/defaultconfigfiles/models/jessee_2018_slim.ini" \
  --lq-model "Zhu 2017:$HOME/gf_output/$EVENT/${EVENT}_zhu_2017_general_slim_model.tif:$HOME/groundfailure/defaultconfigfiles/models/zhu_2017_general_slim.ini" \
  --shakefile ~/shakemap_profiles/default/data/$EVENT/current/products/grid.xml \
  --contours  ~/shakemap_profiles/default/data/$EVENT/current/products/cont_mmi.json \
  --outfile ~/malatya_gf.html 2>&1 | tail -3

In [ ]:
html_path = Path.home() / 'malatya_gf.html'
b64 = base64.b64encode(html_path.read_bytes()).decode()
HTML(f'<iframe src="data:text/html;base64,{b64}" width="100%" height="600px"></iframe>')

**Discussion:**
- Where are the highest landslide probabilities? Does that match your expectation
  based on the terrain?
- Where is liquefaction concentrated? What does that tell you about the
  near-surface geology in those areas?
- How might ground failure affect emergency response — which roads or
  infrastructure might be disrupted?

---
## 6. Try it yourself: Change the fault parameters

Scenario planning involves testing sensitivity to input parameters.
The cells below let you modify the scenario and re-run ShakeMap.

### Option A: Change the magnitude

The M7.2 was estimated from the full 65 km fault length. What if only
part of the fault ruptures? Try M6.5 (roughly a 20 km partial rupture)
and observe how the shaking footprint changes.

In [ ]:
# Read current event.xml
event_xml  = EVENT_DIR / 'event.xml'
orig_xml   = event_xml.read_text()

# Change magnitude — edit the value below
NEW_MAG = '6.5'   # try 6.5, 7.0, or 7.5

import re
modified = re.sub(r'mag="[^"]+"', f'mag="{NEW_MAG}"', orig_xml)
event_xml.write_text(modified)
print(f'Magnitude changed to {NEW_MAG}')

In [ ]:
%%bash
shake $EVENT assemble -c 'magnitude test' model contour mapping 2>&1 \
  | grep -E 'Running|Finished|ERROR'

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
orig_img = Path.home() / 'malatya_global_vs30_intensity.jpg'
for ax, path, title in [
    (axes[0], orig_img,              f'Original M7.2'),
    (axes[1], PRODUCTS/'intensity.jpg', f'Modified M{NEW_MAG}'),
]:
    if Path(str(path)).exists():
        ax.imshow(mpimg.imread(str(path)))
    ax.set_title(title, fontsize=13); ax.axis('off')
plt.suptitle('Effect of magnitude on shaking footprint', fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# Restore original magnitude
event_xml.write_text(orig_xml)
print('Restored to M7.2')

### Option B: Switch to a point source

Remove the fault geometry and run as a point source — same as the
exercise in the San Andreas notebook. How much does the Malatya Fault
geometry change the shaking pattern compared to a simple point source?

In [ ]:
rupture_file   = EVENT_DIR / 'rupture.json'
rupture_backup = EVENT_DIR / 'rupture.json.bak'
shutil.copy(rupture_file, rupture_backup)
rupture_file.unlink()
print('Running as point source...')

In [ ]:
%%bash
shake $EVENT assemble -c 'point source' model contour mapping 2>&1 \
  | grep -E 'Running|Finished|ERROR'

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
orig_img = Path.home() / 'malatya_global_vs30_intensity.jpg'
for ax, path, title in [
    (axes[0], orig_img,               'Finite fault'),
    (axes[1], PRODUCTS/'intensity.jpg','Point source'),
]:
    if Path(str(path)).exists():
        ax.imshow(mpimg.imread(str(path)))
    ax.set_title(title, fontsize=13); ax.axis('off')
plt.suptitle('Finite fault vs. point source — Malatya M7.2', fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# Restore rupture file
shutil.copy(rupture_backup, rupture_file)
rupture_backup.unlink()
print('Restored — back to finite fault')

### Option C: Switch to regional Vs30

The AFAD regional Vs30 map for Turkey captures site conditions at 300 m
resolution — much finer than the global topographic slope model (~1 km).
This matters in urban areas where basin sediments amplify shaking.

In [ ]:
AFAD_VS30  = Path.home() / 'shakemap_data/vs30/afad_turkey_vs30.tif'
model_conf = EVENT_DIR / 'model.conf'
orig_conf  = model_conf.read_text() if model_conf.exists() else ''

if AFAD_VS30.exists():
    model_conf.write_text(orig_conf + f"""
[data]
    vs30filename = {AFAD_VS30}
""")
    print('Switched to AFAD regional Vs30')
else:
    print('AFAD Vs30 not yet available in this image — skip this option')

In [ ]:
%%bash
if [ -f ~/shakemap_data/vs30/afad_turkey_vs30.tif ]; then
    shake $EVENT assemble -c 'AFAD Vs30' model contour mapping 2>&1 \
      | grep -E 'Running|Finished|ERROR'
else
    echo 'AFAD Vs30 not yet available — skip'
fi

In [ ]:
if AFAD_VS30.exists():
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    orig_img = Path.home() / 'malatya_global_vs30_intensity.jpg'
    for ax, path, title in [
        (axes[0], orig_img,               'Global Vs30'),
        (axes[1], PRODUCTS/'intensity.jpg','AFAD regional Vs30'),
    ]:
        if Path(str(path)).exists():
            ax.imshow(mpimg.imread(str(path)))
        ax.set_title(title, fontsize=13); ax.axis('off')
    plt.suptitle('Effect of Vs30 model — Malatya Fault M7.2', fontsize=14)
    plt.tight_layout(); plt.show()

    # restore
    import re
    cleaned = re.sub(r'\[data\].*?vs30filename[^\n]+\n', '',
                     orig_conf + '', flags=re.DOTALL)
    model_conf.write_text(orig_conf)
    print('Restored to global Vs30')

**Discussion:**
- Which Vs30 model produces higher shaking in Malatya city?
- Why might the regional model differ from the global one in river valleys
  and urban basins?
- How would this difference affect ground failure probability estimates?

---
*End of scenario. The next notebook runs the actual 2023 Kahramanmaraş
M7.8 earthquake and compares model results to what was observed.*